In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

print("Baseline Model Development")
print("Step 202")

Baseline Model Development
Step 202


In [3]:
PROJECT_ROOT = Path(
    r"C:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection"
)

PROCESSED_DIR = (
    PROJECT_ROOT /
    "data" /
    "processed" /
    "CWRU"
)

print("Project root:")
print(PROJECT_ROOT)

print("\nProcessed directory:")
print(PROCESSED_DIR)

print("\nExists:", PROCESSED_DIR.exists())

Project root:
C:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection

Processed directory:
C:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU

Exists: True


In [4]:
physics_file = PROCESSED_DIR / "cwru_physics_features.csv"
statistics_file = PROCESSED_DIR / "feature_statistics.csv"

print("Physics file exists:", physics_file.exists())
print("Statistics file exists:", statistics_file.exists())

Physics file exists: True
Statistics file exists: True


In [5]:
physics_df = pd.read_csv(physics_file)
statistics_df = pd.read_csv(statistics_file)

print("Physics features:")
print(physics_df.shape)

print("\nFeature statistics:")
print(statistics_df.shape)

Physics features:
(17, 16)

Feature statistics:
(12, 9)


In [6]:
print("========== PHYSICS FEATURES ==========")
print(physics_df.columns.tolist())

print("\n========== STATISTICS ==========")
print(statistics_df.columns.tolist())

========== PHYSICS FEATURES ==========
['source_file', 'signal_id', 'class', 'RPM', 'RMS', 'Variance', 'Kurtosis', 'Crest_Factor', 'Peak_to_Peak', 'Energy_1x', 'Energy_2x', 'Energy_3x', 'Energy_BPFI', 'Energy_BPFO', 'Energy_FTF', 'Energy_BSF']

========== STATISTICS ==========
['Unnamed: 0', 'count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']


In [7]:
display(physics_df.head())

,source_file,signal_id,class,RPM,RMS,Variance,Kurtosis,Crest_Factor,Peak_to_Peak,Energy_1x,Energy_2x,Energy_3x,Energy_BPFI,Energy_BPFO,Energy_FTF,Energy_BSF
0,B007_0.mat,X118,Ball,1796.0,0.138266,0.018884,2.964506,3.802842,1.046245,12.651480,7.261748,16.516354,5924.331644,12.313566,1.602576,19.271154
1,B007_1.mat,X119,Ball,1772.0,0.138471,0.019157,2.983638,3.720946,1.030164,4.130463,14.695277,6.011213,1880.497997,5.852975,0.413234,46.279088
2,B007_2.mat,X120,Ball,1748.0,0.144331,0.020808,2.724532,3.461846,0.965839,0.962573,12.849932,6.865564,2830.918487,3.090036,0.337600,54.560167
3,B007_3.mat,X121,Ball,1722.0,0.154805,0.023953,2.832997,4.030328,1.233532,0.316885,14.019952,5.584360,5597.967603,6.028661,0.474728,49.992032
4,IR007_0.mat,X105,Inner Race,1797.0,0.289302,0.083467,5.632106,5.477161,2.794696,4.612868,85.396632,9.398079,9877.105213,6.778110,1.214267,74.754989


In [8]:
display(statistics_df.head())

,Unnamed: 0,count,mean,std,min,25%,50%,75%,max
0,RMS,17.0,0.265260,0.212264,0.064424,0.074082,0.154805,0.312346,0.676408
1,Variance,17.0,0.112622,0.149878,0.003967,0.005353,0.023953,0.097539,0.456391
2,Kurtosis,17.0,4.641468,2.078425,2.724532,2.925184,2.983638,5.632106,8.053920
3,Crest_Factor,17.0,4.409673,0.827514,3.461846,3.678552,4.030328,5.209852,5.477161
4,Peak_to_Peak,17.0,2.499756,2.277683,0.439344,0.520287,1.233532,3.037050,6.760144


In [9]:
print("Physics feature rows:", len(physics_df))
print("Window training rows:", len(train_df) if "train_df" in globals() else "not loaded")

Physics feature rows: 17
Window training rows: not loaded


In [10]:
from pathlib import Path
import pandas as pd

WINDOW_DIR = PROJECT_ROOT / "data" / "processed" / "CWRU" / "windows"

TRAIN_METADATA = WINDOW_DIR / "train_metadata.csv"
VAL_METADATA = WINDOW_DIR / "validation_metadata.csv"
TEST_METADATA = WINDOW_DIR / "test_metadata.csv"

train_df = pd.read_csv(TRAIN_METADATA)
val_df = pd.read_csv(VAL_METADATA)
test_df = pd.read_csv(TEST_METADATA)

print("Train metadata:", train_df.shape)
print("Validation metadata:", val_df.shape)
print("Test metadata:", test_df.shape)

print("\nTrain columns:")
print(train_df.columns.tolist())

display(train_df.head())

Train metadata: (390, 7)
Validation metadata: (117, 7)
Test metadata: (76, 7)

Train columns:
['recording_id', 'source_file', 'signal_id', 'class', 'window_id', 'start_sample', 'end_sample']


,recording_id,source_file,signal_id,class,window_id,start_sample,end_sample
0,B007_3_X121,B007_3.mat,X121,Ball,0,0,12000
1,B007_3_X121,B007_3.mat,X121,Ball,1,6000,18000
2,B007_3_X121,B007_3.mat,X121,Ball,2,12000,24000
3,B007_3_X121,B007_3.mat,X121,Ball,3,18000,30000
4,B007_3_X121,B007_3.mat,X121,Ball,4,24000,36000


In [11]:
def get_window_path(row, split):
    recording_id = str(row["recording_id"])
    window_id = int(row["window_id"])

    filename = f"{recording_id}_window_{window_id:04d}.npy"

    return WINDOW_DIR / split / filename


# Test one training file
example_path = get_window_path(train_df.iloc[0], "train")

print("Example window:")
print(example_path)

print("\nExists:", example_path.exists())

Example window:
C:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\windows\train\B007_3_X121_window_0000.npy

Exists: True


In [12]:
CLASS_NAMES = [
    "Healthy",
    "Ball",
    "Inner Race",
    "Outer Race"
]

CLASS_TO_INDEX = {
    name: index
    for index, name in enumerate(CLASS_NAMES)
}

train_df["label"] = train_df["class"].map(CLASS_TO_INDEX)
val_df["label"] = val_df["class"].map(CLASS_TO_INDEX)
test_df["label"] = test_df["class"].map(CLASS_TO_INDEX)

print("Train class distribution:")
print(train_df["class"].value_counts())

print("\nValidation class distribution:")
print(val_df["class"].value_counts())

print("\nTest class distribution:")
print(test_df["class"].value_counts())

Train class distribution:
class
Healthy       276
Inner Race     57
Outer Race     38
Ball           19
Name: count, dtype: int64

Validation class distribution:
class
Healthy       79
Ball          19
Outer Race    19
Name: count, dtype: int64

Test class distribution:
class
Ball          38
Inner Race    19
Outer Race    19
Name: count, dtype: int64
